In [11]:
### Goal: Train a Linear Regression as the first model.
## Evaluate using R² on the test set.
## Record baseline results.

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder as le, OneHotEncoder as One, StandardScaler as ss




In [17]:
df_train = pd.read_csv('train_housing.csv')
df_test = pd.read_csv('test_housing.csv')
print (df_train.head(), df_test.head())
print (df_train.shape, df_test.shape)


drop_cols = [
    'ClosePrice',
    'CloseDate',
    'ListingContractDate',
    'PurchaseContractDate',
    'ContractStatusChangeDate',
    'ListingId',
    'ListingKey',
]

X_train = df_train.drop(columns=drop_cols, errors='ignore')
y_train = df_train['ClosePrice']

X_test = df_test.drop(columns=drop_cols, errors='ignore')
y_test = df_test['ClosePrice']


## check for categorical columns first

cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

bool_like = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 
             'NewConstructionYN', 'FireplaceYN']

true_cat_cols = [col for col in cat_cols if col not in bool_like]

##distinguish boolean-like columns from true categorical columns
print(f"Boolean columns: {bool_like}")
print(f"True categorical columns: {true_cat_cols}")






  ViewYN PoolPrivateYN  OriginalListPrice  ListingKey   CloseDate  ClosePrice  \
0  False         False          1130000.0   538338723  2024-09-12   1090000.0   
1   True          True          1995000.0  1089077716  2024-09-30   1995000.0   
2   True          True          2340000.0  1089076111  2024-09-30   2340000.0   
3   True          True           984000.0  1089075731  2024-09-30    984000.0   
4   True         False          1250000.0  1089075621  2024-09-30   1225000.0   

    Latitude   Longitude PropertyType  LivingArea  ...  Levels  LotSizeArea  \
0  34.136660 -118.012799  Residential      2655.0  ...     One      14390.0   
1  33.804370 -116.439882  Residential      3524.0  ...     One      18295.0   
2  33.515988 -117.706900  Residential      2988.0  ...     Two      11900.0   
3  32.799141 -117.017206  Residential      1573.0  ...     One       6500.0   
4  34.024425 -117.136006  Residential      2889.0  ...     One       5338.0   

  MainLevelBedrooms NewConstructionYN 

/var/folders/_x/x2tsp53j6qnc79kydrrx_hkm0000gn/T/ipykernel_90587/3425802428.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


In [19]:


# boolean-like columns: map True/False to 1/0
# true categorical columns: use LabelEncoder to convert to numeric values
from sklearn.preprocessing import LabelEncoder


for col in bool_like:
    if col in X_train.columns:
        X_train[col] = X_train[col].map({True: 1, False: 1, 
                                          'True': 1, 'False': 0,
                                          '1': 1, '0': 0}).fillna(0)
        X_test[col] = X_test[col].map({True: 1, False: 1,
                                        'True': 1, 'False': 0,
                                        '1': 1, '0': 0}).fillna(0)

## Fit on combined train + test to capture all categories

for col in true_cat_cols:
    combined = pd.concat([X_train[col], X_test[col]]).astype(str).reset_index(drop=True)
    le.fit(combined)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    le_dict[col] = le

print("Encoding done!")
print(X_train.dtypes.value_counts())



TypeError: LabelEncoder.fit() missing 1 required positional argument: 'y'

/var/folders/_x/x2tsp53j6qnc79kydrrx_hkm0000gn/T/ipykernel_81385/1875676180.py:1: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  X.train, X.test, y.train, y.test = train_test_split(X, y, test_size=0.2, random_state=42)
